In [1]:
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
from sklearn.preprocessing import LabelEncoder, StandardScaler

torch.manual_seed(10)
np.random.seed(10)

In [3]:
# 1. Load data
col_names = ['class','age','menopause','tumor-size','inv-nodes','node-caps','deg-malig','breast','breast-quad','irradiat']
df = pd.read_csv('breast+cancer/breast-cancer.data', header=None, names=col_names)
df = df.replace('?', np.nan)
df = df.dropna()
print('Shape:', df.shape)
print(df.head(10))

Shape: (277, 10)
                  class    age menopause tumor-size inv-nodes node-caps  \
0  no-recurrence-events  30-39   premeno      30-34       0-2        no   
1  no-recurrence-events  40-49   premeno      20-24       0-2        no   
2  no-recurrence-events  40-49   premeno      20-24       0-2        no   
3  no-recurrence-events  60-69      ge40      15-19       0-2        no   
4  no-recurrence-events  40-49   premeno        0-4       0-2        no   
5  no-recurrence-events  60-69      ge40      15-19       0-2        no   
6  no-recurrence-events  50-59   premeno      25-29       0-2        no   
7  no-recurrence-events  60-69      ge40      20-24       0-2        no   
8  no-recurrence-events  40-49   premeno      50-54       0-2        no   
9  no-recurrence-events  40-49   premeno      20-24       0-2        no   

   deg-malig breast breast-quad irradiat  
0          3   left    left_low       no  
1          2  right    right_up       no  
2          2   left    left_

In [4]:
# 2. Encode categorical features
le_dict = {}
for col in df.columns:
    le = LabelEncoder()
    df[col] = le.fit_transform(df[col])
    le_dict[col] = le

print(df.head(10))

   class  age  menopause  tumor-size  inv-nodes  node-caps  deg-malig  breast  \
0      0    1          2           5          0          0          2       0   
1      0    2          2           3          0          0          1       1   
2      0    2          2           3          0          0          1       0   
3      0    4          0           2          0          0          1       1   
4      0    2          2           0          0          0          1       1   
5      0    4          0           2          0          0          1       0   
6      0    3          2           4          0          0          1       0   
7      0    4          0           3          0          0          0       0   
8      0    2          2          10          0          0          1       0   
9      0    2          2           3          0          0          1       1   

   breast-quad  irradiat  
0            1         0  
1            4         0  
2            1         0  


In [5]:
# 3. Prepare X, Y and normalize
X = df.iloc[:, 1:].values.astype(np.float32)
Y = df.iloc[:, 0].values.astype(np.float32)

scaler = StandardScaler()
X = scaler.fit_transform(X)

# Y: 0 -> no-recurrence, 1 -> recurrence (giu nguyen 0/1 cho BCELoss)
print('X shape:', X.shape)
print('Y distribution:', np.unique(Y, return_counts=True))

X shape: (277, 9)
Y distribution: (array([0., 1.], dtype=float32), array([196,  81]))


In [6]:
# 4. Train/test split (80/20)
train_size = int(len(X) * 0.8)
print('Training set size:', train_size)

X_train = torch.tensor(X[:train_size], dtype=torch.float32)
Y_train = torch.tensor(Y[:train_size], dtype=torch.float32).unsqueeze(1)

X_test = torch.tensor(X[train_size:], dtype=torch.float32)
Y_test = torch.tensor(Y[train_size:], dtype=torch.float32).unsqueeze(1)

Training set size: 221


In [7]:
# 5. Create neural network
model = nn.Sequential(
    nn.Linear(9, 8),
    nn.ReLU(),
    nn.BatchNorm1d(8),
    nn.Linear(8, 5),
    nn.Tanh(),
    nn.BatchNorm1d(5),
    nn.Linear(5, 1),
    nn.Sigmoid()
)

criterion = nn.BCELoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)

print(model)

Sequential(
  (0): Linear(in_features=9, out_features=8, bias=True)
  (1): ReLU()
  (2): BatchNorm1d(8, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
  (3): Linear(in_features=8, out_features=5, bias=True)
  (4): Tanh()
  (5): BatchNorm1d(5, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
  (6): Linear(in_features=5, out_features=1, bias=True)
  (7): Sigmoid()
)


In [9]:
# 6. Train neural network
epochs = 450
batch_size = 10
n = len(X_train)

for epoch in range(epochs):
    model.train()
    perm = torch.randperm(n)
    epoch_loss = 0.0
    num_batches = 0
    for i in range(0, n, batch_size):
        idx = perm[i:i+batch_size]
        if len(idx) < 2:  # BatchNorm can loi voi batch chi co 1 sample
            continue
        xb = X_train[idx]
        yb = Y_train[idx]

        output = model(xb)
        loss = criterion(output, yb)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        epoch_loss += loss.item()
        num_batches += 1

    if (epoch + 1) % 50 == 0:
        avg_loss = epoch_loss / num_batches
        print(f'Epoch {epoch+1}/{epochs}, Loss: {round(avg_loss, 4)}')

# Evaluate on training set
model.eval()
with torch.no_grad():
    Y_train_pred = model(X_train)
    train_loss = criterion(Y_train_pred, Y_train).item()
    Y_train_cls = (Y_train_pred >= 0.5).float()
    train_acc = (Y_train_cls == Y_train).float().mean().item() * 100

print('Loss - training set:', round(train_loss, 4))
print('Success rate (%) - training set:', round(train_acc, 2))
vals, counts = np.unique(Y_train.numpy(), return_counts=True)
for v, c in zip(vals, counts):
    print(f'  {int(v)}: {c}')

Epoch 50/450, Loss: 0.3011
Epoch 100/450, Loss: 0.2608
Epoch 150/450, Loss: 0.2662
Epoch 200/450, Loss: 0.2405
Epoch 250/450, Loss: 0.2317
Epoch 300/450, Loss: 0.2112
Epoch 350/450, Loss: 0.1769
Epoch 400/450, Loss: 0.1699
Epoch 450/450, Loss: 0.1979
Loss - training set: 0.2385
Success rate (%) - training set: 90.5
  0: 196
  1: 25


In [10]:
# 7. Test neural network
model.eval()
with torch.no_grad():
    Y_test_pred = model(X_test)
    Y_test_cls = (Y_test_pred >= 0.5).float()
    test_acc = (Y_test_cls == Y_test).float().mean().item() * 100

print('Success rate (%) - testing set:', round(test_acc, 2))
vals, counts = np.unique(Y_test.numpy(), return_counts=True)
for v, c in zip(vals, counts):
    print(f'  {int(v)}: {c}')

Success rate (%) - testing set: 8.93
  1: 56
